In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr
from xgboost import XGBRegressor

df = pd.read_csv("f1_ml_ready_dataset.csv", low_memory=False)

print("Dataset shape:", df.shape)

target = "positionOrder"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

model.fit(X_train, y_train)

print("\nModel training completed.")

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\nModel Performance:")
print("MAE:", mae)
print("RMSE:", rmse)

spearman_corr, spearman_p_value = spearmanr(y_test, y_pred)

print("\nSpearman Rank Correlation:")
print("Spearman Correlation:", spearman_corr)
print("P-value:", spearman_p_value)

rank_comparison = pd.DataFrame({
    "Actual_Position": y_test.values,
    "Predicted_Position_Score": y_pred
})

rank_comparison["Actual_Rank"] = rank_comparison["Actual_Position"].rank(
    method="first",
    ascending=True
).astype(int)

rank_comparison["Predicted_Rank"] = rank_comparison["Predicted_Position_Score"].rank(
    method="first",
    ascending=True
).astype(int)

rank_comparison["Rank_Difference"] = (
    rank_comparison["Actual_Rank"] - rank_comparison["Predicted_Rank"]
).abs()

rank_comparison = rank_comparison.sort_values("Actual_Rank")

print("\nActual Rank vs Predicted Rank Table:")
print(rank_comparison.head(20))

rank_comparison.to_csv("actual_vs_predicted_rank.csv", index=False)

print("\nActual vs predicted rank table saved as actual_vs_predicted_rank.csv")

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(importance.head(10))

importance.to_csv("feature_importance.csv", header=["importance"])

print("\nFeature importance saved as feature_importance.csv")

joblib.dump(model, "f1_xgboost_model.pkl")
joblib.dump(X_train.columns.tolist(), "feature_columns.pkl")

print("\nModel saved as f1_xgboost_model.pkl")
print("Feature columns saved as feature_columns.pkl")

Dataset shape: (26759, 53)
Training samples: (21407, 52)
Testing samples: (5352, 52)

Model training completed.

Model Performance:
MAE: 3.3619964122772217
RMSE: 4.4223979735048236

Spearman Rank Correlation:
Spearman Correlation: 0.7760168940603439
P-value: 0.0

Actual Rank vs Predicted Rank Table:
     Actual_Position  Predicted_Position_Score  Actual_Rank  Predicted_Rank  \
54                 1                  2.003498            1             145   
64                 1                  3.196560            2             477   
90                 1                  2.879494            3             378   
126                1                  1.979288            4             138   
205                1                  1.137608            5              12   
207                1                  3.769159            6             620   
208                1                  2.461733            7             270   
217                1                  2.072836            8        

In [2]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [0.1, 1, 10]
}

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=30,                      
    scoring="neg_root_mean_squared_error",
    cv=5,                           
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best score:", search.best_score_)
print("Best parameters:", search.best_params_)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
print("Test RMSE:", rmse)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best score: -4.404485416412354
Best parameters: {'subsample': 0.6, 'reg_lambda': 0.1, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 0.8}
Test RMSE: 4.434154915271011
